# 04 — RAG Customer Chatbot
**NinjaVan Operations Intelligence Suite**

Retrieval-Augmented Generation — Claude API + ChromaDB answers customer queries from the NinjaVan knowledge base.

---
- **Run in:** VSCode with the `ninjavan` conda kernel (`conda activate ninjavan`)
- **Input:** FAQ documents already in `data/rag_documents/` (25 .txt files committed to the repo)
- **Output:** Populated `chroma_db/` vector store + tested RAG pipeline
- **AI types covered:** GenAI (Claude API) + RAG (ChromaDB)

**Before running:** Ensure `ANTHROPIC_API_KEY` is set in `.env` at the repo root.

In [ ]:
import os
from dotenv import load_dotenv

# Load GEMINI_API_KEY from .env (never hardcode keys in notebooks)
load_dotenv()

assert os.environ.get('GEMINI_API_KEY'), \
    'Set GEMINI_API_KEY in .env at the repo root before running this notebook.'
print('Gemini API key loaded from .env')


In [ ]:
from google import genai
import chromadb
import json

print(f'google-genai: {genai.__version__}')
print(f'chromadb: {chromadb.__version__}')


## 1. Create the NinjaVan Knowledge Base

25 synthetic documents covering: parcel tracking, failed delivery, rescheduling, claims, refunds, COD, international shipping, and escalation policies.

In [ ]:
DOCUMENTS = [
    {
        "id": "track-001",
        "topic": "Parcel Tracking",
        "text": """To track your NinjaVan parcel, visit track.ninjavan.co and enter your tracking number.
Your tracking number is printed on the airway bill attached to your parcel and in your shipping
confirmation email. Tracking updates are refreshed every 30 minutes. If your tracking status has
not changed for more than 48 hours, please contact NinjaVan customer support."""
    },
    {
        "id": "track-002",
        "topic": "Tracking Status Meanings",
        "text": """NinjaVan tracking statuses: 'Picked Up' means the parcel has been collected from the seller.
'In Transit' means the parcel is moving between hubs. 'Out for Delivery' means a driver is en route
to your address today. 'Delivered' means the parcel was handed to the recipient or left at a safe
location. 'Delivery Failed' means the driver could not complete the delivery — a reattempt will
be scheduled. 'Return to Sender' means the parcel is being returned after multiple failed attempts."""
    },
    {
        "id": "delivery-001",
        "topic": "Failed Delivery",
        "text": """If your delivery attempt failed, NinjaVan will automatically reattempt delivery on the
next business day. You will receive an SMS and email notification. NinjaVan makes up to 3 delivery
attempts. After 3 failed attempts, the parcel is returned to the sender. To reschedule or provide
a different delivery address, contact customer support within 24 hours of the failed attempt."""
    },
    {
        "id": "delivery-002",
        "topic": "Delivery Time Windows",
        "text": """NinjaVan delivers Monday to Saturday, 9 AM to 10 PM. Same-day delivery is available
in selected cities for orders placed before 12 PM. Standard delivery is 2–5 business days.
Express delivery is 1–2 business days. You can check estimated delivery dates on the tracking page.
Deliveries on public holidays may be delayed by 1–2 business days."""
    },
    {
        "id": "delivery-003",
        "topic": "Safe Drop Policy",
        "text": """NinjaVan drivers may leave your parcel at a safe location (e.g. front door, reception desk,
postbox) if no one is available to receive it. A photo of the parcel placement is taken as proof
of delivery. If you did not receive a parcel that was marked as safe-dropped, contact support
within 48 hours with your tracking number. NinjaVan will investigate and review the delivery photo."""
    },
    {
        "id": "reschedule-001",
        "topic": "Rescheduling Delivery",
        "text": """To reschedule a delivery, visit track.ninjavan.co, enter your tracking number, and click
'Reschedule Delivery'. You can choose a preferred date up to 7 days from the current scheduled date.
Rescheduling must be done at least 2 hours before the current scheduled delivery window.
Each parcel can be rescheduled up to 2 times. For further rescheduling, contact customer support."""
    },
    {
        "id": "reschedule-002",
        "topic": "Change Delivery Address",
        "text": """You can change the delivery address only if the parcel has not yet been assigned to a driver
(status: 'In Transit' or earlier). Contact NinjaVan customer support with your tracking number and
the new delivery address. Address changes within the same city are usually approved within 2 hours.
Cross-city address changes may require seller approval and can take 1–2 business days."""
    },
    {
        "id": "claims-001",
        "topic": "Filing a Damage Claim",
        "text": """To file a damage claim, go to ninjavan.co/claims within 7 days of delivery. You will need:
your tracking number, photos of the damaged item and packaging, and the purchase receipt or invoice.
Claims are reviewed within 5–7 business days. NinjaVan covers damage that occurred during transit.
Damage due to improper packaging by the sender is not covered. You will receive a claim reference
number by email after submission."""
    },
    {
        "id": "claims-002",
        "topic": "Lost Parcel Claim",
        "text": """If your parcel has not been delivered and tracking has not updated for more than 5 business
days, it may be lost. Contact NinjaVan customer support to initiate a lost parcel investigation.
Investigations take 7–14 business days. If the parcel is confirmed lost, NinjaVan will compensate
up to the declared value of the item or SGD 100, whichever is lower, unless additional insurance
was purchased at the time of shipping."""
    },
    {
        "id": "claims-003",
        "topic": "Claims Timeline and Compensation",
        "text": """Damage and loss claims must be submitted within 7 days of the delivery date (or last tracking
update for lost parcels). Approved claims are compensated within 14 business days via bank transfer
or store credit. Standard coverage is up to SGD 100. For higher-value items, NinjaVan offers
declared value protection at an additional fee, which must be arranged with the sender before shipping."""
    },
    {
        "id": "refund-001",
        "topic": "Refund Policy",
        "text": """Refunds are issued by the seller, not NinjaVan. NinjaVan is the delivery partner and does not
process product refunds. For product refunds, contact the seller directly. NinjaVan can assist with
return shipping if the seller has arranged a return. Approved NinjaVan service compensation claims
(e.g. for lost/damaged parcels) are processed separately by NinjaVan's claims team."""
    },
    {
        "id": "cod-001",
        "topic": "Cash on Delivery (COD)",
        "text": """NinjaVan supports Cash on Delivery (COD) for eligible orders. The driver will collect
the exact COD amount at delivery — please prepare the exact amount as drivers may not carry change.
COD is available in SGD, MYR, IDR, THB, PHP, and VND depending on the delivery country.
If you cannot pay the COD amount at delivery, the parcel will be returned to the sender.
Contact the seller to arrange an alternative payment method before the next delivery attempt."""
    },
    {
        "id": "international-001",
        "topic": "International Shipping",
        "text": """NinjaVan offers cross-border shipping within Southeast Asia (Singapore, Malaysia, Indonesia,
Thailand, Philippines, Vietnam). International deliveries take 5–14 business days depending on
the destination. Customs duties and import taxes are the responsibility of the recipient.
Prohibited items (weapons, perishables, live animals, hazardous materials) cannot be shipped.
All international shipments require a customs declaration form completed by the sender."""
    },
    {
        "id": "size-001",
        "topic": "Parcel Size and Weight Limits",
        "text": """NinjaVan accepts parcels up to 30 kg. Maximum dimensions: 150 cm on the longest side,
and a combined girth + length of no more than 300 cm. Parcels exceeding these limits require
a special arrangement — contact business@ninjavan.co. Oversized or overweight parcels may be
returned to sender if not pre-arranged. Fragile items must be clearly labelled and adequately packed."""
    },
    {
        "id": "contact-001",
        "topic": "Contact NinjaVan Support",
        "text": """NinjaVan customer support is available 7 days a week, 8 AM to 10 PM local time.
Contact options: Live Chat at ninjavan.co (fastest response, typically under 5 minutes during
business hours), Email at support@ninjavan.co (response within 1 business day), Phone at
+65 6018 9009 (Singapore) or your local NinjaVan number. Always have your tracking number ready."""
    },
    {
        "id": "driver-001",
        "topic": "Driver Behaviour Complaints",
        "text": """To report a concern about a NinjaVan driver (e.g. unprofessional behaviour, damaged parcel
at handover, unsafe driving), contact support@ninjavan.co with your tracking number, the date and
time of the incident, and a description of what occurred. All reports are reviewed within 3 business
days. NinjaVan takes driver conduct seriously — verified violations result in retraining or
disciplinary action."""
    },
    {
        "id": "proof-001",
        "topic": "Proof of Delivery",
        "text": """NinjaVan captures proof of delivery (POD) for every parcel: either a recipient signature or
a photo of the parcel at the delivery location. To request your POD, contact support with your
tracking number. POD photos are retained for 90 days. If you dispute a delivery, NinjaVan will
share the POD as part of the investigation. Sellers can access POD via the NinjaVan merchant portal."""
    },
    {
        "id": "pickup-001",
        "topic": "Parcel Pickup Points",
        "text": """If you missed a delivery, you can arrange to collect your parcel from a NinjaVan hub or
partner pickup point. Request a hold at your nearest hub via the tracking page or customer support.
Parcels are held for up to 5 business days. Bring your tracking number and a government-issued ID
for collection. Hub locations are listed at ninjavan.co/locations."""
    },
    {
        "id": "returns-001",
        "topic": "Return Shipments",
        "text": """To return a parcel to the seller, the seller must arrange a return pickup via the NinjaVan
merchant portal. You will receive a return tracking number and a scheduled pickup time via SMS.
Ensure the item is securely re-packaged before the driver arrives. NinjaVan does not accept
returns without a pre-arranged return order from the seller. Contact the seller first to initiate
the return process."""
    },
    {
        "id": "sla-001",
        "topic": "Delivery SLA and Delays",
        "text": """NinjaVan's standard delivery SLA is 2–5 business days from pickup. Express SLA is 1–2
business days. Delays may occur during peak periods (11.11, 12.12, year-end), public holidays,
or severe weather. During peak periods, standard SLA may extend by 1–3 business days. NinjaVan
will proactively notify recipients of any significant delays via SMS and email."""
    },
    {
        "id": "account-001",
        "topic": "NinjaVan Account and Login",
        "text": """To track parcels you do not need a NinjaVan account — tracking is accessible via
track.ninjavan.co with just your tracking number. Sellers and businesses can register for a
NinjaVan merchant account at ship.ninjavan.co. For account login issues, use the 'Forgot Password'
link or contact support@ninjavan.co. NinjaVan will never ask for your password via email or chat."""
    },
    {
        "id": "prohibited-001",
        "topic": "Prohibited Items",
        "text": """NinjaVan does not ship the following: weapons and ammunition, live animals, perishable
food without proper packaging, flammable or hazardous materials, narcotics and controlled substances,
counterfeit goods, and items prohibited by destination country customs. Attempting to ship prohibited
items may result in parcel seizure, return to sender, or legal action. When in doubt, contact
support before shipping."""
    },
    {
        "id": "escalate-001",
        "topic": "Escalation Process",
        "text": """If your issue has not been resolved after 3 business days, you can request escalation
to a NinjaVan senior case manager. To escalate: reply to your existing support ticket with the
subject 'ESCALATE', include your case reference number, and describe the unresolved issue.
Senior case managers respond within 1 business day. For urgent escalations, call the NinjaVan
priority line at +65 6018 9009 and press 2."""
    },
    {
        "id": "notification-001",
        "topic": "Delivery Notifications",
        "text": """NinjaVan sends automated SMS and email notifications at key delivery milestones: when the
parcel is picked up from the seller, when it is out for delivery, when delivery is successful, and
when a delivery attempt fails. To update your contact details for notifications, contact the seller
who placed the shipment. NinjaVan cannot update recipient contact details directly."""
    },
    {
        "id": "merchant-001",
        "topic": "For Merchants and Businesses",
        "text": """Businesses that ship more than 50 parcels per month can apply for a NinjaVan business
account with volume pricing, API integration, and a dedicated account manager. Apply at
ship.ninjavan.co/business. NinjaVan's API supports order creation, real-time tracking webhooks,
proof of delivery retrieval, and bulk shipment uploads. Contact business@ninjavan.co for
enterprise SLA requirements and custom integrations."""
    },
]

print(f'Knowledge base ready: {len(DOCUMENTS)} documents')
print(f'Topics: {[d["topic"] for d in DOCUMENTS]}')

## 2. Save Documents to `data/rag_documents/`

Persist the FAQ text files so they can be version-controlled and re-indexed without re-running this notebook.

In [ ]:
import pathlib

RAG_DIR = pathlib.Path('data/rag_documents')
RAG_DIR.mkdir(parents=True, exist_ok=True)

for doc in DOCUMENTS:
    filepath = RAG_DIR / f"{doc['id']}.txt"
    with open(filepath, 'w') as f:
        f.write(f"Topic: {doc['topic']}\n\n{doc['text']}")

print(f'{len(DOCUMENTS)} documents saved to {RAG_DIR}/')
print('Files:', sorted([p.name for p in RAG_DIR.glob('*.txt')]))

## 3. Index into ChromaDB

ChromaDB stores documents as embeddings in a local vector database. When a user asks a question, ChromaDB finds the most semantically similar documents and passes them to Claude as context.

In [ ]:
CHROMA_PATH      = './chroma_db'
COLLECTION_NAME  = 'ninjavan_kb'

chroma = chromadb.PersistentClient(path=CHROMA_PATH)

# Delete and recreate to ensure a clean index
try:
    chroma.delete_collection(COLLECTION_NAME)
    print('Cleared existing collection.')
except Exception:
    pass

collection = chroma.create_collection(
    name=COLLECTION_NAME,
    metadata={'hnsw:space': 'cosine'},
)

collection.add(
    ids       = [d['id'] for d in DOCUMENTS],
    documents = [d['text'] for d in DOCUMENTS],
    metadatas = [{'topic': d['topic']} for d in DOCUMENTS],
)

print(f'Indexed {collection.count()} documents into ChromaDB.')
print(f'Stored at: {CHROMA_PATH}')

## 4. Test the RAG Pipeline (Retrieval Only)

Verify ChromaDB retrieves the correct documents before involving Claude.

In [ ]:
def retrieve(query: str, n: int = 3) -> list:
    results = collection.query(query_texts=[query], n_results=n)
    docs = results['documents'][0]
    meta = results['metadatas'][0]
    return list(zip(meta, docs))


TEST_QUERIES = [
    'Where is my parcel?',
    'The driver left my package outside and it got stolen',
    'How do I reschedule my delivery to next week?',
    'My item arrived damaged, how do I claim compensation?',
]

for q in TEST_QUERIES:
    print(f'\nQ: {q}')
    hits = retrieve(q, n=2)
    for meta, _ in hits:
        print(f'  → {meta["topic"]}')

## 5. Full RAG Pipeline with Claude

Retrieve top-k documents → build context → ask Claude to answer strictly from context.

In [ ]:
import os

_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
_MODEL = "gemini-2.5-flash"

SYSTEM_PROMPT = """You are a NinjaVan customer service assistant.
Answer the customer's question using ONLY the context provided.
Be concise, friendly, and helpful.
If the answer is not in the context, say:
  'I do not have enough information to answer that. I will escalate this to a human agent.'
Do not make up information not found in the context."""

def rag_answer(query: str, n_docs: int = 3) -> dict:
    hits    = retrieve(query, n=n_docs)
    context = "\n\n".join([text for _, text in hits])
    topics  = [meta["topic"] for meta, _ in hits]

    full_prompt = f"{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nCustomer question: {query}"
    response = _client.models.generate_content(model=_MODEL, contents=full_prompt)
    answer   = response.text
    escalated = "escalate" in answer.lower() or "human agent" in answer.lower()

    return {"query": query, "answer": answer, "sources": topics, "escalated": escalated}


In [ ]:
# Run test queries
TEST_CASES = [
    'Where is my parcel? Tracking number NV12345.',
    'My delivery failed, when will they try again?',
    'I want to reschedule my delivery to Saturday.',
    'My item arrived damaged. What do I do?',
    'Can I change my delivery address?',
    'What time does the driver deliver?',
    'The driver did not ring my doorbell.',
    'Can I ship a laptop overseas?',
    'How do I contact NinjaVan?',
    'What happens if parcel is lost?',
]

results = []
for q in TEST_CASES:
    r = rag_answer(q)
    results.append(r)
    print(f'Q: {q}')
    print(f'A: {r["answer"][:120]}...' if len(r['answer']) > 120 else f'A: {r["answer"]}')
    print(f'   Sources: {r["sources"]} | Escalated: {r["escalated"]}')
    print()

## 6. Evaluation

Manual QA metrics: answer rate (how often it answered vs escalated) and source relevance.

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
answered   = (~df['escalated']).sum()
escalated  = df['escalated'].sum()
deflection = answered / len(df) * 100

print('=== RAG Chatbot Evaluation ===')
print(f'Total test queries:    {len(df)}')
print(f'Answered by AI:        {answered} ({deflection:.0f}%)')
print(f'Escalated to human:    {escalated}')
print(f'Deflection rate:       {deflection:.0f}%')

print('\nPer-query summary:')
print(df[['query', 'escalated', 'sources']].to_string(index=False))

## 7. Hallucination Guard Test

Verify the chatbot correctly refuses to answer questions outside the knowledge base.

In [ ]:
OUT_OF_SCOPE = [
    'What is the best restaurant near Orchard Road?',
    'Can you write me a Python script?',
    'What is the stock price of NinjaVan?',
]

print('=== Hallucination Guard Tests ===')
for q in OUT_OF_SCOPE:
    r = rag_answer(q)
    print(f'Q: {q}')
    print(f'Escalated: {r["escalated"]} | Answer: {r["answer"][:100]}...')
    print()

## 8. Commit Knowledge Base to GitHub

In [ ]:
# Model saved — commit to GitHub so HF Spaces has it:
#   git add src/models/
#   git commit -m 'Add trained model'
#   git push
print("Done. Commit and push src/models/ to GitHub.")

## 9. ChromaDB Setup Script

Since `chroma_db/` is gitignored, it must be recreated at deployment time (HF Spaces, Modal).  
Run this whenever you need to rebuild the vector store from the committed text files.

## Summary

| Component | Detail |
|-----------|--------|
| Knowledge base | 25 NinjaVan FAQ documents |
| Vector DB | ChromaDB (cosine similarity) |
| LLM | Claude Sonnet 4.6 |
| Retrieval | Top-3 documents per query |
| Deflection rate | see above |
| Hallucination guard | Escalates when context is irrelevant |

**The knowledge base text files** are committed to `data/rag_documents/`.  
**ChromaDB** is rebuilt from those files at app startup — see `src/utils/chroma_setup.py`.

**Next:** The `src/agents/customer_agent.py` is already wired up.  
Run `app/streamlit_app.py` to demo the full chatbot in the UI.